# Project Overview

This notebook preprocesses the RetentionAI dataset for binary classification. It loads the data, validates quality, encodes features, scales numeric inputs, and performs a train/test split.

# Import Libraries

Group all imports for the preprocessing workflow.

In [ ]:
from pathlib import Path
import sys

import joblib
import numpy as np
import pandas as pd

# Ensure the project root is on sys.path so `src` imports work in notebooks
workspace_root = Path.cwd()
while not (workspace_root / "src").exists() and workspace_root != workspace_root.parent:
    workspace_root = workspace_root.parent
sys.path.insert(0, str(workspace_root.resolve()))

from src.data.load_data import load_dataset
from src.features.cleaning import (
    convert_total_charges,
    remove_missing_values,
    remove_duplicates,
    reset_dataframe_index,
)
from src.features.encoding import encode_target, one_hot_encode
from src.features.split import split_dataset
from src.features.scaling import scale_features
from src.data.save_data import save_dataframe

# Reload encoding module in-case it was modified during this session
import importlib
import src.features.encoding as _enc_mod
importlib.reload(_enc_mod)
from src.features.encoding import encode_target, one_hot_encode


# Load Dataset

Load the dataset and display the first rows and the dataset shape.

In [48]:
data_path = Path("..") / "data" / "raw" / "customer_churn.csv"

# Reload the module to ensure we have the latest function definition in the kernel
import importlib
import src.data.load_data as load_mod
importlib.reload(load_mod)

# Call the reloaded function
df = load_mod.load_dataset(str(data_path))

# Quick preview
df.head()
print(df.shape)


(7043, 21)


# Dataset Validation

Validate the loaded dataset structure before preprocessing.

In [49]:
df.info()
print(f"Current shape: {df.shape}")

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

# Convert TotalCharges to Numeric

Convert `TotalCharges` to numeric values and inspect the data type before and after conversion.

In [50]:
print(f"Current data type: {df['TotalCharges'].dtype}")

df = convert_total_charges(df)

print(f"New data type: {df['TotalCharges'].dtype}")


Current data type: str
New data type: float64


# Missing Values Analysis

Review missing values by count and percentage.

In [51]:
missing_counts = df.isna().sum()
missing_summary = pd.DataFrame({
    'missing_count': missing_counts,
    'missing_percentage': (missing_counts / len(df) * 100).round(2),
}).sort_values('missing_count', ascending=False)

display(missing_summary)


,missing_count,missing_percentage
TotalCharges,11,0.16
gender,0,0.00
SeniorCitizen,0,0.00
Partner,0,0.00
customerID,0,0.00
Dependents,0,0.00
tenure,0,0.00
MultipleLines,0,0.00
PhoneService,0,0.00
OnlineSecurity,0,0.00


# Handle Missing Values

Drop rows with missing values and reset the index.

In [52]:
rows_before = len(df)
df = remove_missing_values(df)
rows_removed = rows_before - len(df)

print(f"Rows removed: {rows_removed}")
print(f"Updated dataset shape: {df.shape}")


Rows removed: 11
Updated dataset shape: (7032, 21)


# Duplicate Validation

Check for duplicate rows after handling missing values.

In [53]:
duplicate_rows = df.duplicated().sum()
print(f"Duplicate rows: {duplicate_rows}")

df = remove_duplicates(df)
print(f"Duplicate rows after removal: {df.duplicated().sum()}")


Duplicate rows: 0
Duplicate rows after removal: 0


# Categorical Feature Detection

Detect categorical columns while excluding identifiers and the target.

In [54]:
target_column = 'Churn'
excluded_columns = ['customerID', target_column]

categorical_features = (
    df.select_dtypes(include=['object', 'category', 'bool']).columns
    .drop(excluded_columns, errors='ignore')
    .tolist()
)

print('Categorical features:', categorical_features)


Categorical features: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


C:\Users\ascom\AppData\Local\Temp\ipykernel_484\3376026265.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df.select_dtypes(include=['object', 'category', 'bool']).columns


# One-Hot Encoding

One-hot encode the identified categorical features and report the dataset shape before and after encoding.

In [55]:
shape_before_encoding = df.shape

# Encode the target before one-hot encoding categorical features
df = encode_target(df)

# One-hot encode categorical features
shape_before_encoding = df.shape
df = one_hot_encode(df)
shape_after_encoding = df.shape

print(f"Dataset shape before encoding: {shape_before_encoding}")
print(f"Dataset shape after encoding: {shape_after_encoding}")


Dataset shape before encoding: (7032, 21)
Dataset shape after encoding: (7032, 7062)


# Prepare Target Variable

Convert the target column to binary numeric labels.

In [56]:
target_column = 'Churn'

df = encode_target(df)

print('Unique target values after conversion:', df[target_column].unique())
print(df[target_column].value_counts().sort_index())


Unique target values after conversion: [nan]
Series([], Name: count, dtype: int64)


# Split Features and Target

Separate features and the prediction target.

In [57]:
X = df.drop(columns=[target_column])
y = df[target_column].copy()

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")


X shape: (7032, 7061)
y shape: (7032,)


# Train/Test Split

Split the dataset while preserving the target distribution.

In [58]:
X_train, X_test, y_train, y_test = split_dataset(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

print("\nTarget distribution before split:")
print(y.value_counts(normalize=True).sort_index())

print("\nTarget distribution in y_train:")
print(y_train.value_counts(normalize=True).sort_index())

print("\nTarget distribution in y_test:")
print(y_test.value_counts(normalize=True).sort_index())


ValueError: Input y contains NaN.

# Feature Scaling

Scale numeric features using a scaler fitted only on the training set.

In [ ]:
X_train_scaled, X_test_scaled, scaler = scale_features(X_train, X_test)

artifacts_dir = Path('artifacts')
scaler_path = artifacts_dir / 'scaler.pkl'
artifacts_dir.mkdir(parents=True, exist_ok=True)
joblib.dump(scaler, scaler_path)

print(f"X_train_scaled shape: {X_train_scaled.shape}")
print(f"X_test_scaled shape: {X_test_scaled.shape}")
print(f"Saved StandardScaler to {scaler_path.resolve()}")


X_train_scaled shape: (5625, 31)
X_test_scaled shape: (1407, 31)
Saved StandardScaler to D:\cv projects\RetentionAI\notebooks\artifacts\scaler.pkl


# Final Validation

Verify the final preprocessing outputs before moving to modeling.

In [ ]:
validation_results = []

validation_results.append(("No missing values in X_train_scaled", X_train_scaled.isna().sum().sum() == 0))
validation_results.append(("No missing values in X_test_scaled", X_test_scaled.isna().sum().sum() == 0))
validation_results.append(("No missing values in y_train", y_train.isna().sum() == 0))
validation_results.append(("No missing values in y_test", y_test.isna().sum() == 0))
validation_results.append(("No object columns remain in X_train_scaled", X_train_scaled.select_dtypes(include=['object']).shape[1] == 0))
validation_results.append(("X_train_scaled and X_test_scaled have identical columns", X_train_scaled.columns.equals(X_test_scaled.columns)))
validation_results.append(("Target is binary", set(y_train.unique()) <= {0, 1} and set(y_test.unique()) <= {0, 1}))

for message, passed in validation_results:
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {message}")

print("\nFinal dataset shapes:")
print(f"X_train_scaled: {X_train_scaled.shape}")
print(f"X_test_scaled: {X_test_scaled.shape}")
print(f"y_train: {y_train.shape}")
print(f"y_test: {y_test.shape}")


[PASS] No missing values in X_train_scaled
[PASS] No missing values in X_test_scaled
[PASS] No missing values in y_train
[PASS] No missing values in y_test
[FAIL] No object columns remain in X_train_scaled
[PASS] X_train_scaled and X_test_scaled have identical columns
[PASS] Target is binary

Final dataset shapes:
X_train_scaled: (5625, 31)
X_test_scaled: (1407, 31)
y_train: (5625,)
y_test: (1407,)


C:\Users\ascom\AppData\Local\Temp\ipykernel_484\2739380553.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  validation_results.append(("No object columns remain in X_train_scaled", X_train_scaled.select_dtypes(include=['object']).shape[1] == 0))


# Save Preprocessed Dataset

Save the fully preprocessed dataset before train/test split so the modeling workflow can consume a stable, production-ready input.

In [ ]:
from pathlib import Path

processed_dir = Path("..") / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)
processed_file = processed_dir / "preprocessed_retention.csv"

preprocessed_df = pd.concat([X, y], axis=1)
preprocessed_df.to_csv(processed_file, index=False)

print(f"Saved preprocessed dataset to: {processed_file.resolve()}")
print(f"Dataset shape: {preprocessed_df.shape}")


Saved preprocessed dataset to: D:\cv projects\RetentionAI\data\processed\preprocessed_retention.csv
Dataset shape: (7032, 32)


Saving the processed dataset creates a clear boundary between data preparation and model development. It also makes the modeling notebook faster to run and easier to reproduce without re-running preprocessing.

# Save Pipeline Outputs

Save the train/test split outputs so the modeling workflow can consume the exact data partitions generated by preprocessing.

In [ ]:
pipeline_output_dir = processed_dir / "pipeline_outputs"
pipeline_output_dir.mkdir(parents=True, exist_ok=True)

x_train_file = pipeline_output_dir / "X_train.csv"
x_test_file = pipeline_output_dir / "X_test.csv"
y_train_file = pipeline_output_dir / "y_train.csv"
y_test_file = pipeline_output_dir / "y_test.csv"

X_train.to_csv(x_train_file, index=False)
X_test.to_csv(x_test_file, index=False)
y_train.to_csv(y_train_file, index=False)
y_test.to_csv(y_test_file, index=False)

print(f"Saved pipeline outputs to: {pipeline_output_dir.resolve()}")
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")


Saved pipeline outputs to: D:\cv projects\RetentionAI\data\processed\pipeline_outputs
X_train shape: (5625, 31)
X_test shape: (1407, 31)
y_train shape: (5625,)
y_test shape: (1407,)


Saving the train/test split outputs ensures reproducibility by fixing the exact data partitions used for training and evaluation. It also keeps the modeling notebook independent from preprocessing, so model experiments can be rerun without regenerating data splits.